### 1.1.2. Indexing Expressions

An **indexing expression** is a set expression enclosed in braces `{...}` that scopes the iteration of a model component or operator over a set. In standard mathematical notation, phrases such as 
- for all *p* ∈ PROD, or
- for all  *j* ∈ FOOD such that *c*[*j*] > 0 

are written in AMPL as indexing expressions.

Indexing expressions appear in:
- **Parameter and variable declarations** — specifying the index set
- **Constraint and objective declarations** — specifying which instances to generate
- **Reduction operators** (`sum`, `prod`, `min`, `max`) — specifying the summation domain
- **Set definitions** — defining a derived set by comprehension
- **Interactive commands** (`display`, `let`) and amplpy (`get_data`) — filtering entities post-solve

#### Formal Syntax

The grammar of an indexing expression is:
```python
indexing:
    {sexpr-list}
    {sexpr-list: lexpr }
Where:
    sexpr-list:
        sexpr                   # anonymous: set expression only
        dummy-member 'in' sexpr # named: dummy index bound to set
        sexpr-list, sexpr       # multiple sets (cross-product)
    lexpr:
        logical condition referencing dummy indices, sets, and parameters
```
Optional: The `lexpr` within an indexing expression selects only those elements that satisfy the logical expression, excluding the others. `lexpr` typically involves one or more dummy indices of the indexing expression.    lexpr:

| Syntax | Meaning | Example |
|--------|---------|---------|
| `{idx in set}` | Named indexing | `var x {p in PROD} >= 0;` |
| `{idx in 1..T}` | Named numeric range (T is a parameter) | `var x {t in 1..T} >= 0;` |
| `{set}` or <br> `{1..param}` | Anonymous indexing | `var x {PROD} >= 0;` or <br>`var x {1..T} >= 0;`|
| `{idx in set: condition}` or <br>`{idx in 1..param: condition}`| Filtered indexing | `{j in FOOD: cost[j] > 0}` or  <br>`{j in 1..T: cost[j] > 0}`|
| `{idx1 in set1, idx2 in set2}` or <br>`{idx1 in set1, idx2 in 1..param}`| Cross product | `{p in PROD, i in NODES}`or <br>`{p in PROD, t in 1..T}`|

#### Dummy Indices

A **dummy index** is a local variable introduced by `dummy-member in sexpr` within an indexing expression. Its definition is effective only within the **scope** of the indexing expression that defines it.

**Scope rules**:
1. The scope of a **dummy index**
- **Begins:** Immediately after the set expression that defines it
- **Ends:**
    - For declaration-level indexing (param, var, constraint): at the end of the enclosing statement
    - For reduction operators (`sum`, `min`, `max`, `prod`): at the end of the operand

In [ ]:
### DECLARATION-LEVEL INDEXING: p is in scope for the entire variable or constraint definition
# ✅ A) Parameter declaration
param Sell {p in PROD} >= Min_sell[p] <= Max_sell[p];
#     └──────────────────────┬──────────────────────┘
#                 Scope of dummy index p

# ✅ B) Constraint declaration (p is in scope throughout the constraint)
subject to MinSales {p in PROD}: Sell[p] >= 0.1 * market[p];
#          └───────────────────────┬───────────────────────┘
#                       Scope of dummy index p

# ✅ C) Variable Declaration
var Make {p in PROD} >= 0, <= capacity[p];
#    └─────────────────┬─────────────────┘
#            Scope of dummy index p


### REDUCTION OPERATOR
# ✅ Objective: Each sum creates an independent scope - the two p indices are unrelated
maximize TotalRevenue:
    sum {p in PROD} revenue[p] * Sell[p] - sum {j in PROD} cost[j] * Sell[j];
#   └────────────────┬─────────────────┘   └────────────────┬───────────────┘
#       Scope of the first sum operand       Scope of the second sum operand

# ✅ Equivalent compact form (p is in scope for the entire expression within parentheses.)
maximize TotalRevenue: sum {p in PROD} (revenue[p] - cost[p]) * Sell[p];
#                      └───────────────────────┬───────────────────────┘   
#                                  Scope of the sum operand

2. A nested indexing expression introduces its own dummy indices; those inner indices **shadow** any outer index of the same name within the nested scope.

In [ ]:
# Nested index shadows outer index of the same name
subject to FoodRatio {j in FOOD}: Buy[j] >= min_frac[j] * sum {i in FOOD} Buy[i];
#          |                                              └──────────┬──────────|
#          |                                        Scope of inner dummy index i|
#          └──────────────────────────────────┬─────────────────────────────────┘ 
#                                Scope of outer dummy index j

3. Once the scope ends, the **dummy index** is undefined. The same name may be reused freely in subsequent statements — it does not accumulate state between statements.

In [ ]:
# Same name reused across separate statements — no state carried over
subject to MinSales {p in PROD}: Sell[p] >= 0;          # p is local to this statement
subject to MaxSales {p in PROD}: Sell[p] <= market[p];  # p is a fresh index here, unrelated to above   

maximize TotalRevenue:
    sum {p in PROD} revenue[p] * Sell[p] - sum {p in PROD} cost[p] * Sell[p];
#   └────────────────┬─────────────────┘   └────────────────┬───────────────┘
#     Scope of the first dummy index p      Scope of the second dummy index p

A common convention is to associate a fixed index name with a fixed set (`i` for `NUTR`, `j` for `FOOD`). This is a readability convention only; AMPL imposes no such restriction.

**Choosing distinct names for nested indexing.** If the outer scope's dummy index is `j`, a nested indexing expression over the same set must use a different name (e.g., `i`) to avoid shadowing:

In [ ]:
# j is in scope for the whole constraint body;
# i is introduced for the inner sum only
subject to FoodRatio {j in FOOD}: Buy[j] >= min_frac[j] * sum {i in FOOD} Buy[i];

#### Anonymous Indexing

When the body of the indexed construct does not reference the index value, the **dummy index** name may be omitted. The indexing expression then consists of the set expression alone:

```python
{PROD}         — anonymous; equivalent to {p in PROD} when p is not used
{1..T}         — anonymous numeric range
```

The two declarations below are semantically equivalent; the second uses anonymous indexing because the constraint body does not use the index name directly:

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__1.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
# Named form — p is referenced in the bound attribute
param cost  {p in PROD} >= min_cost[p];   # lower bound depends on p
param price {p in PROD} in [cost[p], max_price[p]];   # bounds reference p

# Anonymous form — the attribute does not reference the iterated member
param cost  {PROD} >= 0;    # same bound applies to every member; p not needed
param price {PROD} >= 0;
param units {PROD} integer >= 1;

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use a named <b>dummy index</b> whenever the index value appears anywhere in the statement body — in subscripts, arithmetic expressions, or conditions. Reserve anonymous indexing for the rare case where iteration count matters but not the value (e.g., <code>sum {SHIFTS} fixed_cost</code>).</div>

#### Numeric Range Indexing

AMPL supports arithmetic range expressions of the form `first..last` (and `first..last by step`) as set expressions. These ranges are **ordered sets** of integers (or reals, when a step is specified).

A scalar parameter may appear as either endpoint:

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__2.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
param T = 5;                           # number of time periods
param cost{1..T} = 10;                 # anonymous range: creates cost[1]=10, cost[2]=10, ..., cost[5]=10
param price {t in 1..T by 2} = cost[t] + 5;   # named range with step 2: t = 1,3,5 only
                                              # t is available to index cost[t]
display cost, price;                   # Shows: cost[1..5]=10, price[1]=15, price[3]=15, price[5]=15

**When to use the named form.** Use `{t in 1..T}` whenever the body must reference the current index value — subscripting variables or parameters, computing relative indices such as `t-1`, or writing filtered conditions:

In [ ]:
subject to Balance {t in 1..T}: inventory[t] = inventory[t-1] + produce[t] - demand[t];
#                                                        ^^^
#                              t-1 is only possible because t is a named dummy index

**`T` is a parameter, not a set.** The expression `1..T` is evaluated at model-generation time using the current value of `T`. Changing `T` via a `let` or `data` command and re-solving regenerates the correct number of instances automatically.

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__3.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
param T integer > 0;        # Number of time periods (parameter, not a set)

var x {1..T} >= 0;          # Decision variable indexed by 1..T

minimize Total:             # Objective: sum over dynamically created index set
    sum {t in 1..T} x[t];

#### Conditional (Filtered) Indexing

A colon `:` followed by a logical expression (`lexpr`) restricts the indexing expression to only those elements for which the condition is true. The condition may reference:
- Set membership tests (`i in MAXREQ`)
- Arithmetic comparisons on parameters (`n_min[i] > 0`)
- Already-defined dummy indices from the same indexing expression
- Boolean combinations using `and`, `or`, `not`

Within a **model declaration**, the condition must be a constant expression — it may not reference variables or constraints (their values are not defined at model-generation time).

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__4.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
# Filtered derived set: nutrients in MAXREQ or with positive minimum
set NUTREQ = {i in NUTR: i in MAXREQ or n_min[i] > 0};

# Filtered sum: cost of foods above their minimum amount
param excess_cost = sum {j in FOOD: f_min[j] > 0} cost[j] * f_min[j];

# Filtered constraint: only for foods with narrow allowable range
subject to NarrowRange {j in FOOD: f_max[j] - f_min[j] < 1}:
    Buy[j] = f_min[j];

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Variables and constraints may <em>not</em> appear in the condition of an indexing expression inside a model declaration. For example, <code>{j in FOOD: Buy[j] > 0}</code> is illegal inside a <code>subject to</code> or <code>param</code> declaration. Variables are only defined after the model is generated; conditions in declarations must be evaluable at model-generation time from sets and parameters alone.</div>

#### Indexing over Empty Sets

Indexing over an **empty set** is never an error in AMPL. The semantics are well-defined:

- A model component (parameter, variable, constraint) declared over an empty set has no instances — AMPL silently generates zero components.
- A `sum` over an empty set evaluates to **0** (the additive identity).
- A `prod` over an empty set evaluates to **1** (the multiplicative identity).
- `min` over an empty set evaluates to `Infinity`; `max` evaluates to `-Infinity`.
- `forall` over an empty set is vacuously **true**; `exists` over an empty set is **false**.

Example: if no nutrient has both a minimum and maximum requirement, `BOTHREQ` is empty and the constraint `DualLimit` simply does not exist in the generated model — no error is raised.

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__5.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
set MINREQ;  # nutrients with a lower bound
set MAXREQ;  # nutrients with an upper bound

# BOTHREQ is empty when MINREQ and MAXREQ are disjoint
set BOTHREQ = {i in MINREQ: i in MAXREQ};

# This constraint generates zero instances if BOTHREQ is empty — not an error
subject to DualLimit {i in BOTHREQ}: sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Design models so that empty index sets produce correct, not merely error-free, results. A constraint indexed over an empty set is trivially satisfied — verify that this is intentional. A <code>sum</code> over an empty set returns zero — verify that zero is the correct default for the expression's role in the model.</div>

#### Scope Rules: Model Declarations vs. Interactive Commands

The allowable content of an indexing expression's filter condition depends on context:

| Context | May reference |
|---|---|
| Model declaration (`param`, `var`, `subject to`, `set =`) | Sets, parameters, dummy indices |
| Interactive command (`display`, `let`) | Sets, parameters, dummy indices, variables, constraints |
| amplpy `get_data`, `get_value` | Sets, parameters, dummy indices, variables, constraints |

In [ ]:
# Interactive AMPL command — Buy[j] is a variable, legal post-solve
display {j in FOOD: Buy[j] > f_min[j]};

#### ⚠️ Edge Case 1: Reusing the Same Index Name in Nested Expressions 

❌ Problem

In AMPL, you cannot reuse the same dummy index name within a nested indexing expression.

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__6.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
# ❌ BUGGY (do not use): incorrect double indexing over FACTORIES
subject to Capacity {f in FACTORIES}:
    sum {f in FACTORIES, p in PRODUCTS} produce[f,p] <= capacity[f];

🔴 Result

``` python
[ERROR] syntax error
	context:  sum {f  >>> in  <<< FACTORIES, p in PRODUCTS} produce[f,p] <= capacity[f];
```
**Why this happens**?
- The constraint already defines an index: `subject to Capacity {f in FACTORIES}`
- Inside the `sum`, we try to declare `f` again: `sum {f in FACTORIES, p in PRODUCTS}`
AMPL does not allow redeclaring the same index name in the same expression scope

✅ **Correct approaches**: Use the outer index properly: No duplication of index names

In [ ]:
subject to Capacity {f in FACTORIES}:
    sum {p in PRODUCTS} produce[f,p] <= capacity[f];

**Edge case 2: Multi-index with a set that depends on the first index**

The second set expression in a multi-index may reference a dummy index defined by the first. This allows iteration over structures where the valid values for one index depend on another — for example, arc lists in a network or time-dependent activity sets.

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__6.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
set NODES;
set ARCS {NODES};   # ARCS[i] = set of nodes reachable from node i

# (i,j) pairs where j is in ARCS[i] — not the full cross-product NODES x NODES
param flow_cap {i in NODES, j in ARCS[i]} >= 0;

# This generates only the declared arcs, not all node pairs
var Flow {i in NODES, j in ARCS[i]} >= 0, <= flow_cap[i,j];

**Edge case 3: Indexed set comprehension yielding an empty set**

A filtered derived set may be empty for some data instances. Any constraint or parameter indexed over that derived set will simply have no instances — but the declaration itself is valid. This is the correct way to model optional structure.

<a href="https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__6.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
set PROD;
param cost {PROD} >= 0;

# EXPENSIVE is empty when no product costs more than 100
set EXPENSIVE = {p in PROD: cost[p] > 100};

# If EXPENSIVE is empty, no instances of this constraint are generated
subject to PremiumCap {p in EXPENSIVE}: Make[p] <= 500;